In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 94.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/Qwen/Qwen2.5-3B-Instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt1 = "\nPhotorealistic full-length image of a Vietnamese áo dài displayed without a human model.\n\nOverall impression: Elegant, graceful, refined Vietnamese cultural aesthetics.\n\nGarment Details:\nColor: deep blue\nFabric: brocade\nSilhouette: loose relaxed silhouette\nFloral Embellishment: embroidered floral motifs\nNeckline: high mandarin collar\nSleeves: long fitted sleeves\nWaist & Closure: decorative buckle detail\nLength & Hem: mid-calf tunic\n\nSetting:\nBackground: cultural performance stage\nLighting: shade with subtle highlights\n"
prompt2 = "\nPhotorealistic full-length image of a Vietnamese áo dài displayed without a human model.\n\nOverall impression: Elegant, graceful, refined Vietnamese cultural aesthetics.\n\nGarment Details:\nColor: baby blue\nFabric: silk\nSilhouette: layered panel style\nFloral Embellishment: embroidered floral motifs\nNeckline: U-shaped neckline\nSleeves: elbow-length sleeves\nWaist & Closure: hidden zipper closure\nLength & Hem: knee-length tunic\n\nSetting:\nBackground: historic temple steps\nLighting: overcast gentle lighting\n"

prompt_template_1 = f"""You are given a structured fashion description of a Vietnamese áo dài (garment-only, no human features).
Your task is to generate a short, concise image caption suitable for an image–text retrieval dataset starting with "A photo of"

Rules:

Output one single sentence

Keep it neutral, factual, and descriptive

Focus on garment appearance, material, color, silhouette, and pattern

Mention cultural context only if explicitly present

Do not mention mannequins, displays, lighting techniques, or photography terms

Avoid adjectives that imply emotion or narrative

Use clear fashion vocabulary

Input:
{prompt1}

Output:
A short caption describing the áo dài as seen in the image starting with A photo of"""

prompt_template_2 = f"""You are given a structured fashion description of a Vietnamese áo dài (garment-only, no human features).
Your task is to generate a short, concise image caption suitable for an image–text retrieval dataset.

Rules:

Output one single sentence

Keep it neutral, factual, and descriptive

Focus on garment appearance, material, color, silhouette, and pattern

Mention cultural context only if explicitly present

Do not mention mannequins, displays, lighting techniques, or photography terms

Avoid adjectives that imply emotion or narrative

Use clear fashion vocabulary

Input:
{prompt2}

Output:
A short caption describing the áo dài as seen in the image starting with A photo of.
"""

message1 = [
		 {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
		 {"role": "user", "content": prompt_template_1}
]

message2 = [
		 {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
		 {"role": "user", "content": prompt_template_2}
]

messages = [message1, message2]

text = [tokenizer.apply_chat_template(
    msg,
    tokenize=False,
    add_generation_prompt=True
) for msg in messages]

model_inputs = tokenizer(text, return_tensors="pt", padding = True).to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
# generated_ids = [
#     output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
# ]

# response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
# print(response)
# ----- remove prompt tokens -----
generated_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]

# ----- decode ALL outputs -----
responses = tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

for i, r in enumerate(responses):
    print(f"Output {i}: {r.strip()}")


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Output 0: A photo of a deep blue Vietnamese áo dài made of brocade fabric, featuring a loose relaxed silhouette with long fitted sleeves and high mandarin collar, adorned with embroidered floral motifs, showcasing a decorative buckle at the waist, set against a cultural performance stage background with subtle highlights.
Output 1: A photo of a baby blue silk áo dài with a U-shaped neckline and elbow-length sleeves, featuring embroidered floral motifs and a layered panel style, worn over historic temple steps under overcast gentle lighting.


# Inference code

In [ ]:
from tqdm import tqdm
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

batch_size = 1
prompts_path = "/content/new_prompts_extra.json"
output_path = "/content/new_output_extra.json"
output = {}
# with open(prompts_path, "r", encoding = "utf-8") as f:
#     prompts = json.load(f)

prompts = {
   "6994": "\nPhotorealistic full-length image of a Vietnamese áo dài displayed without a human model.\n\nOverall impression: Elegant, graceful, refined Vietnamese cultural aesthetics.\n\nGarment Details:\nColor: deep blue\nFabric: velvet\nSilhouette: modern fitted silhouette\nFloral Embellishment: beading / sequins floral accents\nNeckline: U-shaped neckline\nSleeves: raglan sleeves\nWaist & Closure: sash / belt closure\nLength & Hem: ankle-length tunic\n\nSetting:\nBackground: traditional courtyard\nLighting: early morning sun\n",
   "6995": "\nPhotorealistic full-length image of a Vietnamese áo dài displayed without a human model.\n\nOverall impression: Elegant, graceful, refined Vietnamese cultural aesthetics.\n\nGarment Details:\nColor: baby blue\nFabric: linen\nSilhouette: layered panel style\nFloral Embellishment: beading / sequins floral accents\nNeckline: boat / open neckline\nSleeves: flared sleeves\nWaist & Closure: hidden zipper closure\nLength & Hem: floor-length tunic with side slits\n\nSetting:\nBackground: traditional courtyard\nLighting: overcast gentle lighting\n",
   "6996": "\nPhotorealistic full-length image of a Vietnamese áo dài displayed without a human model.\n\nOverall impression: Elegant, graceful, refined Vietnamese cultural aesthetics.\n\nGarment Details:\nColor: red\nFabric: organza\nSilhouette: asymmetric hem silhouette\nFloral Embellishment: beading / sequins floral accents\nNeckline: U-shaped neckline\nSleeves: short sleeves\nWaist & Closure: hidden zipper closure\nLength & Hem: high side slit variant\n\nSetting:\nBackground: cultural performance stage\nLighting: diffused outdoor lighting\n",
   "6997": "\nPhotorealistic full-length image of a Vietnamese áo dài displayed without a human model.\n\nOverall impression: Elegant, graceful, refined Vietnamese cultural aesthetics.\n\nGarment Details:\nColor: red\nFabric: lace\nSilhouette: asymmetric hem silhouette\nFloral Embellishment: beading / sequins floral accents\nNeckline: rounded neckline\nSleeves: cap sleeves\nWaist & Closure: elastic cinched waist\nLength & Hem: asymmetric hem tunic\n\nSetting:\nBackground: classic architecture scene\nLighting: golden hour lighting\n",
   "6998": "\nPhotorealistic full-length image of a Vietnamese áo dài displayed without a human model.\n\nOverall impression: Elegant, graceful, refined Vietnamese cultural aesthetics.\n\nGarment Details:\nColor: baby blue\nFabric: silk\nSilhouette: layered panel style\nFloral Embellishment: beading / sequins floral accents\nNeckline: low modest V-neckline\nSleeves: elbow-length sleeves\nWaist & Closure: elastic cinched waist\nLength & Hem: asymmetric hem tunic\n\nSetting:\nBackground: quiet urban street\nLighting: soft natural daylight\n",
   "6999": "\nPhotorealistic full-length image of a Vietnamese áo dài displayed without a human model.\n\nOverall impression: Elegant, graceful, refined Vietnamese cultural aesthetics.\n\nGarment Details:\nColor: lavender\nFabric: organza\nSilhouette: layered panel style\nFloral Embellishment: beading / sequins floral accents\nNeckline: rounded neckline\nSleeves: long fitted sleeves\nWaist & Closure: diagonal snap closure\nLength & Hem: short tunic with trousers\n\nSetting:\nBackground: riverside promenade\nLighting: overcast gentle lighting\n"
}
print(f"Number of prompts: {len(prompts)}")
for i in tqdm(range(len(prompts) // batch_size), desc = "Inference"):
  prompts_list = dict(list(prompts.items())[i * batch_size : (i + 1) * batch_size])
  prompts_template_list = [
  f"""You are given a structured fashion description of a Vietnamese áo dài (garment-only, no human features).
  Your task is to generate a short, concise image caption suitable for an image–text retrieval dataset starting with "A photo of"

  Rules:

  Output one single sentence starting with A photo of (strictly).
  Keep it neutral, factual, and descriptive (strictly)
  Focus on garment appearance, material, color, silhouette, and pattern
  Mention cultural context only if explicitly present
  Do not mention mannequins, displays, lighting techniques, or photography terms
  No adjectives that imply emotion or narrative
  Use clear fashion vocabulary

  Input:
  {prompt}

  Output:
  A short caption describing the áo dài as seen in the image"""
  for prompt in prompts_list.values()]
  messages = [
      [
          {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
          {"role": "user", "content": prompt_template}
      ] for prompt_template in prompts_template_list
    ]
  text = [tokenizer.apply_chat_template(
    msg,
    tokenize=False,
    add_generation_prompt=True
  ) for msg in messages]

  model_inputs = tokenizer(text, return_tensors="pt", padding = True).to(model.device)

  generated_ids = model.generate(
      **model_inputs,
      max_new_tokens=512
  )
  # ----- remove prompt tokens -----
  generated_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]

  # ----- decode ALL outputs -----
  responses = tokenizer.batch_decode(
      generated_ids,
      skip_special_tokens=True
  )

  for index, r in enumerate(responses):
      output[list(prompts_list.keys())[index]] = r.strip()
      print(f"Output {list(prompts_list.keys())[index]}: {r.strip()}")
with open(output_path, "w") as f:
  json.dump(output, f, indent = 4)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Number of prompts: 6


Inference:  17%|█▋        | 1/6 [00:04<00:20,  4.19s/it]

Output 6994: A photo of an elegant, modern-fitted Vietnamese áo dài in deep blue velvet with U-shaped neckline and raglan sleeves adorned with beading and sequin floral accents, cinched at the waist with a sash, worn in an early morning light against a backdrop of a traditional courtyard.


Inference:  33%|███▎      | 2/6 [00:07<00:14,  3.67s/it]

...

Output 6997: A photo of an elegant Vietnamese áo dài in red lace with floral beading accents, asymmetrical hem, rounded neckline, cap sleeves, elastic cinched waist, and a tunic length, set against a backdrop of classic architecture under golden hour lighting.


Inference:  83%|████████▎ | 5/6 [00:17<00:03,  3.43s/it]

Output 6998: A photo of an elegant Vietnamese áo dài in baby blue silk featuring a layered panel style, low modest V-neckline, elbow-length sleeves, and beaded floral accents, with an asymmetrical hem and elastic cinched waist, displayed on a quiet urban street under soft natural daylight.


Inference: 100%|██████████| 6/6 [00:20<00:00,  3.37s/it]

Output 6999: A photo of an elegant lavender organza áo dài with floral beading and long fitted sleeves, featuring a layered panel silhouette and a diagonal snap closure, displayed on a riverside promenade under overcast gentle lighting.


In [ ]:
# 3h inferencing. Qwen2.5-3B-Instruct with batch-size 26.

In [ ]:
path = "/content/new_output.json"
with open(path, "r", encoding = "utf-8") as f:
  output = json.load(f)
with open("/content/final_new_captions.json", "w", encoding = "utf-8") as f:
  json.dump(output, f, indent = 4)